Importing Libraries

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve
import numpy as np

Read & View Data

In [3]:
email_spam_dataset = pd.read_csv('email_spam_dataset.csv')
email_spam_dataset.head()

,email_text,label
0,Here is the project update you asked for.,ham
1,Limited offer!!! Buy now and get 50% discount.,spam
2,Win cash prizes instantly by replying to this ...,spam
3,Urgent! Your account has been suspended. Verif...,spam
4,"Hi, please find the meeting agenda attached.",ham


In [4]:
email_spam_dataset.isna().sum()

email_text    0
label         0
dtype: int64

In [5]:
email_spam = pd.read_csv('email_spam.csv', encoding='latin-1')
email_spam = email_spam.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)
email_spam.columns = ['label', 'email_text']
email_spam = email_spam [['email_text', 'label']]
email_spam.head()

,email_text,label
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham


In [6]:
email_spam.isna().sum()

email_text    0
label         0
dtype: int64

In [7]:
# combine the two datasets to create a larger dataset for training the model

df = pd.concat([email_spam_dataset, email_spam], ignore_index=True)

In [8]:
df.head()

,email_text,label
0,Here is the project update you asked for.,ham
1,Limited offer!!! Buy now and get 50% discount.,spam
2,Win cash prizes instantly by replying to this ...,spam
3,Urgent! Your account has been suspended. Verif...,spam
4,"Hi, please find the meeting agenda attached.",ham


Pre-process

In [9]:
def preprocess_text(text):
    # 1. Lowercase 
    text = text.lower()
    # 2. Tokenize
    tokens = word_tokenize(text)
    # 3. Filter out punctuation (keep only alphanumeric tokens)
    tokens = [t for t in tokens if t.isalnum()]
    # 4. Remove stop words
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    # 5. Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

df['clean_tokens'] = df['email_text'].apply(preprocess_text)
df.head()

,email_text,label,clean_tokens
0,Here is the project update you asked for.,ham,"[project, update, asked]"
1,Limited offer!!! Buy now and get 50% discount.,spam,"[limited, offer, buy, get, 50, discount]"
2,Win cash prizes instantly by replying to this ...,spam,"[win, cash, prize, instantly, replying, email]"
3,Urgent! Your account has been suspended. Verif...,spam,"[urgent, account, suspended, verify, immediately]"
4,"Hi, please find the meeting agenda attached.",ham,"[hi, please, find, meeting, agenda, attached]"


In [10]:
# join the cleaned tokens back into a single string for vectorization
df['processed_string'] = df['clean_tokens'].apply(lambda x: ' '.join(x))
df.head()

,email_text,label,clean_tokens,processed_string
0,Here is the project update you asked for.,ham,"[project, update, asked]",project update asked
1,Limited offer!!! Buy now and get 50% discount.,spam,"[limited, offer, buy, get, 50, discount]",limited offer buy get 50 discount
2,Win cash prizes instantly by replying to this ...,spam,"[win, cash, prize, instantly, replying, email]",win cash prize instantly replying email
3,Urgent! Your account has been suspended. Verif...,spam,"[urgent, account, suspended, verify, immediately]",urgent account suspended verify immediately
4,"Hi, please find the meeting agenda attached.",ham,"[hi, please, find, meeting, agenda, attached]",hi please find meeting agenda attached


In [11]:
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(df['processed_string'])
print(bow_matrix.shape)

(5892, 7497)


In [12]:
# create a DataFrame from the BoW matrix
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=vectorizer.get_feature_names_out())

# TF-IDF Vectorization
tfidf_vec = TfidfVectorizer(ngram_range=(1,2))
tfidf_matrix = tfidf_vec.fit_transform(df['processed_string'])

# create a DataFrame from the TF-IDF matrix
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vec.get_feature_names_out())

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    tfidf_matrix, 
    df['label'], 
    test_size=0.2,   # 20% for testing
    random_state=42  # For consistent results
)

In [14]:
# # 1. Initialize the model
# model = MultinomialNB()

# # 2. Train it (the 'fitting' phase)
# model.fit(X_train, y_train)

# # 3. Make predictions on the test set
# y_pred = model.predict(X_test)

In [15]:
# model = RandomForestClassifier(
#     n_estimators=100,
#     random_state=42,
#     class_weight="balanced"
# )

# model.fit(X_train, y_train)

# y_pred = model.predict(X_test)

In [16]:
param_grid = {
    "C": [0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs", "saga"],
    "penalty": ["l1", "l2"],
    "max_iter": [500, 1000, 2000]
}
model = LogisticRegression(class_weight="balanced", random_state=42)

In [17]:
grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="recall",
    n_jobs=-1,
    verbose=1
)

grid.fit(tfidf_matrix, df['label'])

Fitting 5 folds for each of 72 candidates, totalling 360 fits


c:\Users\raeve\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
60 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
60 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\raeve\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\raeve\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\raeve\AppData\Local\Programs\Python\Python39\lib\site-packages\s

GridSearchCV(cv=5,
             estimator=LogisticRegression(class_weight='balanced',
                                          random_state=42),
             n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100], 'max_iter': [500, 1000, 2000],
                         'penalty': ['l1', 'l2'],
                         'solver': ['liblinear', 'lbfgs', 'saga']},
             scoring='recall', verbose=1)

In [18]:
print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Params: {'C': 0.1, 'max_iter': 500, 'penalty': 'l1', 'solver': 'liblinear'}
Best Score: nan


In [19]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

In [20]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[947  53]
 [ 45 134]]
              precision    recall  f1-score   support

         ham       0.95      0.95      0.95      1000
        spam       0.72      0.75      0.73       179

    accuracy                           0.92      1179
   macro avg       0.84      0.85      0.84      1179
weighted avg       0.92      0.92      0.92      1179



In [21]:
y_pred = (best_model.predict_proba(X_test)[:,1] > 0.3)

In [22]:
THRESHOLD = 0.4

def analyze_with_threshold(user_input):
    # Apply SAME preprocessing as training
    clean_input = " ".join(preprocess_text(user_input))
    
    vectorized_input = tfidf_vec.transform([clean_input])
    
    probs = best_model.predict_proba(vectorized_input)[0]
    spam_prob = probs[1]
    
    if spam_prob >= THRESHOLD:
        label = "🚨 SPAM"
    else:
        label = "✅ HAM"
        
    print(f"Result: {label} (Spam Score: {spam_prob:.2f})")

In [23]:
print("Spam Detector AI initialized. (Type 'quit' to exit)")
while True:
    user_text = input("\nEnter a message to scan: ")
    print(f"User Input: {user_text}")
    if user_text.lower() == 'quit':
        break
    analyze_with_threshold(user_text)

Spam Detector AI initialized. (Type 'quit' to exit)
User Input: quit


In [24]:
import joblib

joblib.dump(best_model, 'model.pkl')

['model.pkl']